In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [2]:
path_file = '~/Bureau/exports/20241112/AGS_20241112_exports_agronomes_-archive/AGS_20241112_exports_agronomes_assolees_synthetisees.csv'

ENTREPOT_PATH = '~/Bureau/utils/data/'
df = {}

#### Import des données

In [3]:
# ------------------ #
# IMPORT DES DONNÉES #
# ------------------ #


def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'recolte_rendement_prix', 
    'destination_valorisation',
    'action_realise_agrege', 
    'action_synthetise_agrege'
]

tables_without_id = [
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 4/4 [00:32<00:00,  8.08s/it]
0it [00:00, ?it/s]


In [10]:
df['destination_valorisation'].loc[
    df['destination_valorisation']['code_destination_a'].str.contains('MARA')
]

,code_destination_a,libelle,code_espece_botanique,code_qualifiant_aee,libelle_espece,valorisation_vin,filiere,rendement_unite,source
id,,,,,,,,,
fr.inra.agrosyst.api.entities.referential.RefDestination_0531fce9-fdcf-4450-904d-546c87bb8ec4,MARA_00012,Toutes catégories,ZAL,NaN,Aubergine,NaN,MARAICHAGE,TONNE_HA,Agrosyst_2016
fr.inra.agrosyst.api.entities.referential.RefDestination_80c790bf-151f-4f18-82cb-c8d0f4860c66,MARA_00013,Toutes catégories,ZAL,NaN,Aubergine,NaN,MARAICHAGE,UNITE_HA,Agrosyst_2016
fr.inra.agrosyst.api.entities.referential.RefDestination_fa6547b5-8e1e-4507-bbbc-36a2f8f6b98c,MARA_00093,Alimentation humaine_tous calibres,ZOD,NaN,Pois chiche,NaN,MARAICHAGE,TONNE_HA,Agrosyst_2016
fr.inra.agrosyst.api.entities.referential.RefDestination_cad9cd40-e9b0-4920-b22f-79da063133f9,MARA_00033,Toutes catégories,ZBB,NaN,Chou,NaN,MARAICHAGE,TONNE_HA,Agrosyst_2016
fr.inra.agrosyst.api.entities.referential.RefDestination_d1327dc3-e576-4ac6-9260-25731e9f9d64,MARA_00117,Toutes catégories,NaN,NaN,Toutes,NaN,MARAICHAGE,TONNE_HA,Agrosyst_2018
...,...,...,...,...,...,...,...,...,...
fr.inra.agrosyst.api.entities.referential.RefDestination_c5870087-81c9-41ca-8e10-6fca13653f34,MARA_00002,A compléter,NaN,NaN,Toutes,NaN,MARAICHAGE,TONNE_HA,Agrosyst_2016
fr.inra.agrosyst.api.entities.referential.RefDestination_3d8193e8-c6f3-4659-a63c-fd85e31210c0,MARA_00003,A compléter,NaN,NaN,Toutes,NaN,MARAICHAGE,KG_M2,Agrosyst_2016
fr.inra.agrosyst.api.entities.referential.RefDestination_62e4a657-1fbe-4cf0-96fe-47a1bc02ef64,MARA_00004,A compléter,NaN,NaN,Toutes,NaN,MARAICHAGE,UNITE_HA,Agrosyst_2016


In [25]:
left = df['recolte_rendement_prix']
right = df['destination_valorisation'][['libelle']]
df['recolte_rendement_prix_extanded'] = pd.merge(left, right, left_on = 'destination_id', right_index=True, how='left')

left = df['recolte_rendement_prix_extanded']
right = df['action_realise_agrege'][['noeuds_realise_id', 'plantation_perenne_phases_realise_id']]
df['recolte_rendement_prix_extanded_realise'] = pd.merge(left, right, left_on = 'action_id', right_index=True, how='inner')

left = df['recolte_rendement_prix_extanded']
right = df['action_synthetise_agrege'][['connection_synthetise_id', 'plantation_perenne_phases_synthetise_id']]
df['recolte_rendement_prix_extanded_synthetise'] = pd.merge(left, right, left_on = 'action_id', right_index=True, how='inner')

df['recolte_rendement_prix_extanded_realise']['itk_id'] = df['recolte_rendement_prix_extanded_realise']['noeuds_realise_id'].fillna(df['recolte_rendement_prix_extanded_realise']['plantation_perenne_phases_realise_id'])
df['recolte_rendement_prix_extanded_synthetise']['itk_id'] = df['recolte_rendement_prix_extanded_synthetise']['connection_synthetise_id'].fillna(df['recolte_rendement_prix_extanded_synthetise']['plantation_perenne_phases_synthetise_id'])

In [ ]:
df['recolte_rendement_prix_extanded'] = pd.concat([
    df['recolte_rendement_prix_extanded_synthetise'][['rendement_moy', 'rendement_unite', 'destination', 'itk_id']],
    df['recolte_rendement_prix_extanded_realise'][['rendement_moy', 'rendement_unite', 'destination', 'itk_id']]
])

In [ ]:
import pandas as pd
import numpy as np
import warnings


# ──────────────────────────────────────────────────────────────────────
# Dictionnaire : libellés pertinents extraits de "test.csv"
# ──────────────────────────────────────────────────────────────────────
categories = {
    'grain': [
        'Grain (ethanol)',
        'Grain (biscuiterie)',
        'Grain (amidon)',
        'Grain (alimentation humaine)',
        'Grain (alimentation animale)',
        'Grain',
        'Grain (oléagineux)',
        'Grain (semoule et pâtes alimentaires)',
        'Grain (meunerie)',
        'Grain (malterie)',
        'Grain (industrie divers)'
    ],

    'paille': [
        'Paille'
    ],

    'fourrage': [
        'Fourrage (enrubannage)',
        'Fourrage (ensilage)',
        'Fourrage (distribution en frais)',
        'Fourrage (foin)'
    ],

    'sucre': [
        'Sucre et dérivés (t de sucre)',
        'Sucre et dérivés (t de racines à 16% de richesse)',
        'Sucre et dérivés (t de biomasse en matière sèche)'
    ],

    'fibre': [
        'Fibre'
    ],

    'semences': [
        'Production semences'
    ],

    'bioenergie': [
        'Bioénergie'
    ],

    'ttes_categ': [
        'A compléter',
        'Toutes catégories - tous calibres',
        'Toutes catégories',
        'Tous conditionnements',
        'Tous calibres',
        'Toutes catégories - tous conditionnements',
        'Tabac',
        'Gros. 1/2 gros. producteurs. paysagistes – Tige gros calibre',
        'Gros. 1/2 gros. producteurs. paysagistes – Tige standard',
        'Sans label / Toutes catégories',
        'Toutes catégories - tous calibres - tous conditionnements',
        'Sans appellation / Tous modes de commercialisation',
        'Vente directe - Pot 1l',
        'Primeur / Toutes catégories - tous calibres',
        'Fraiches / Toutes catégories - tous calibres',
        'Vente directe  - Pot 3l',
        'Frais / Tous conditionnements',
        'Vente directe - Pleine terre',
        'Toutes appellations / Tous modes de commercialisation',
        'Alimentation humaine_tous calibres',
        'Gros. 1/2 gros. producteurs. paysagistes - Pleine terre',
        'Gros. 1/2 gros. producteurs. paysagistes – Pot 5l (Conteneur)',
        'Gros. 1/2 gros. producteurs. paysagistes – Pot 1l',
        'Gros. 1/2 gros. producteurs. paysagistes - Pot 3l',
        'Vente directe - Godet (0.6l)',
        'Vente directe – Suspension / Coupe / jardinière (7.5l)',
        'Gros. 1/2 gros. producteurs. paysagistes - Conteneur 15l',
        'Gros. 1/2 gros. producteurs. paysagistes - Godet (0.6l)',
        'Gros. 1/2 gros. producteurs. paysagistes - Suspension / Coupe / jardinière (7.5l)',
        'Vente directe – Pot 5l (conteneur)',
        'Transformation / Toutes catégories - tous calibres',
        'Distillerie',
        'Olivier / Huile d\'olive AOP Provence',
        'Olivier / Huile d\'olive de France',
        "Olivier / Olives de Table",
        'Circuit long / appellation Lorraine',
        'Categ. I',
        'Crue / Tous calibres',
        'Categ. Extra',
        'Circuit Long',
        'Industrie',
        'Circuit Court',
        'NE PAS SAISIR - Toutes catégories - tous calibres',
        'Olivier / Huile d\'olive aromatisée',
        'Sucrerie',
        'Fraiches / Categ. II - tous calibres',
        'Fraiches / Categ. extra - tous calibres',
        'Sans appellation / Vente directe (bouteille)',
        'Epis',
        'A écosser / Toutes catégories',
        'Frais / Toutes catégories - tous calibres',
        'Toutes formes/couleurs - toutes catégories',
        'Feuilles / Toutes catégories',
        'Sans label / Toutes catégories - tous calibres',
        'Circuit Long / Frais',
        'Circuit Long / Transformation',
        'Circuit Court / Transformation',
        'Eau-de-vie',
        'Circuit Court / Frais',
        'Compost',
        'Sans appellation / Négoce (vente_vin)',
        'Sans appellation / Cave coopérative (vente_ raisins)',
        'Exportation / Frais',
        'Exportation / Transformation'
    ]
}

def aggr_rend_by_cat(group: pd.DataFrame) -> pd.Series:
    """
    Calcule, pour chaque catégorie définie dans `categories`,
    la moyenne des rendements et l’unité correspondante.
    """
    out = {}
    df = group[['rendement_moy', 'rendement_unite', 'destination']].copy()

    for cat, libelles in categories.items():
        sub = df[df['destination'].isin(libelles)]

        # Si aucune ligne pour cette catégorie on laisse NaN
        if sub.empty:
            out[f'{cat}_rend_mean'] = np.nan
            out[f'{cat}_unit']      = np.nan
            continue

        uniq_units = sub['rendement_unite'].unique()
        if len(uniq_units) == 1:
            # si l'unité est unique pour la catégorie :
            out[f'{cat}_unit'] = uniq_units[0]
            out[f'{cat}_rend_mean'] = sub['rendement_moy'].mean()
        else:
            # Plusieurs unités : impossible de calculer une moyenne
            # cohérente sans conversion préalable.
            out[f'{cat}_unit'] = 'MULTIPLE'
            out[f'{cat}_rend_mean'] = np.nan
    return pd.Series(out)



In [39]:
import numpy as np
import pandas as pd

# 1. Mapper chaque destination vers sa catégorie globale
dest_to_cat = {lib: cat for cat, libelles in categories.items() for lib in libelles}

sub = df['recolte_rendement_prix_extanded'][
    ['rendement_moy', 'rendement_unite', 'destination', 'itk_id']
].copy()

# 2. Associer la catégorie directement dans le DataFrame
sub['category'] = sub['destination'].map(dest_to_cat)

# On conserve uniquement les lignes qui appartiennent à une catégorie connue
sub_filtered = sub.dropna(subset=['category'])

# 3. Agrégation par (itk_id, category) en une seule passe
grouped = sub_filtered.groupby(['itk_id', 'category']).agg(
    rend_mean=('rendement_moy', 'mean'),
    unit_unique=('rendement_unite', 'nunique'),
    unit_first=('rendement_unite', 'first')
).reset_index()

# 4. Gérer le cas des unités multiples
grouped['unit'] = np.where(
    grouped['unit_unique'] == 1, 
    grouped['unit_first'], 
    'MULTIPLE'
)
grouped['rend_mean'] = np.where(
    grouped['unit_unique'] == 1, 
    grouped['rend_mean'], 
    np.nan
)

# 5. Pivoter pour obtenir exactement le format d'origine (1 colonne par cat_rend_mean et cat_unit)
pivot_mean = grouped.pivot(index='itk_id', columns='category', values='rend_mean')
pivot_mean.columns = [f'{c}_rend_mean' for c in pivot_mean.columns]

pivot_unit = grouped.pivot(index='itk_id', columns='category', values='unit')
pivot_unit.columns = [f'{c}_unit' for c in pivot_unit.columns]

# 6. Combiner les résultats
result = pd.concat([pivot_mean, pivot_unit], axis=1)

# Réordonner les colonnes pour chaque catégorie comme dans votre code initial (optionnel)
cols_order = []
for cat in categories.keys():
    if f'{cat}_rend_mean' in result.columns:
        cols_order.extend([f'{cat}_rend_mean', f'{cat}_unit'])

result = result.reindex(columns=cols_order).reset_index()

In [50]:
result.sample(10)['itk_id'].values

array(['fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_77b7434f-5361-4e3f-8913-87d013fb0238',
       'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_9ee3a5d1-702b-40da-a730-5c8f63a44690',
       'fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_48038484-ff59-472f-a374-da244c618515',
       'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_9cc936d0-33cc-4193-8b20-244a6e78871d',
       'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_5204361f-b2b0-406d-81e8-45836b7cf8cd',
       'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_0481c255-6bf7-4ece-851a-1746dc26bfff',
       'fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_d7ceb81f-6450-4e9b-a98d-ef24c1eafb19',
       'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_1d10c59d-ac38-4c91-9332-5c6db59c8104',
       'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_5e042ad2-07ce-468e-97a0-b45a2

In [56]:
result.loc[
    result['itk_id'].isin([
        'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_49d34aeb-4faa-4148-bf97-695d2bc19104',
        'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_273b56df-fb85-4aa9-8722-b5ff19e8e95d',
        'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_77b7434f-5361-4e3f-8913-87d013fb0238',
        'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_ebb0bb5e-8d8c-4f37-bf0f-620bcd625783',
        'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_7bcbbf3d-7dda-4f1a-985d-b573f39454e4',
        'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_1d10c59d-ac38-4c91-9332-5c6db59c8104'
    ])
]

,itk_id,grain_rend_mean,grain_unit,paille_rend_mean,paille_unit,fourrage_rend_mean,fourrage_unit,sucre_rend_mean,sucre_unit,fibre_rend_mean,fibre_unit,semences_rend_mean,semences_unit,bioenergie_rend_mean,bioenergie_unit,ttes_categ_rend_mean,ttes_categ_unit
17322,fr.inra.agrosyst.api.entities.effective.Effect...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.800000,TONNE_HA
32443,fr.inra.agrosyst.api.entities.effective.Effect...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.774667,TONNE_HA
52862,fr.inra.agrosyst.api.entities.effective.Effect...,33.0,Q_HA_TO_STANDARD_HUMIDITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124151,fr.inra.agrosyst.api.entities.practiced.Practi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.000000,TONNE_HA
153769,fr.inra.agrosyst.api.entities.practiced.Practi...,55.0,Q_HA_TO_STANDARD_HUMIDITY,4.0,TONNE_MS_HA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188563,fr.inra.agrosyst.api.entities.practiced.Practi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.000000,TONNE_HA


In [ ]:
result

,itk_id,level_1,0
0,fr.inra.agrosyst.api.entities.effective.Effect...,grain_unit,Q_HA_TO_STANDARD_HUMIDITY
1,fr.inra.agrosyst.api.entities.effective.Effect...,grain_rend_mean,115.0
2,fr.inra.agrosyst.api.entities.effective.Effect...,paille_unit,TONNE_MS_HA
3,fr.inra.agrosyst.api.entities.effective.Effect...,paille_rend_mean,1.0
4,fr.inra.agrosyst.api.entities.effective.Effect...,fourrage_rend_mean,NaN
...,...,...,...
3442765,test1,semences_unit,NaN
3442766,test1,bioenergie_rend_mean,NaN
3442767,test1,bioenergie_unit,NaN
3442768,test1,ttes_categ_rend_mean,NaN


In [ ]:
dict_map = {
    'grain' : 
    'paille' : 
    

}

In [54]:
df['destination_valorisation']['libelle'].value_counts().to_csv('test.csv')

In [ ]:
df['recolte_rendement_prix_extanded'].samp

,libelle_culture,commercialisation_pct,autoconsommation_pct,nonvalorisation_pct,rendement_min,rendement_max,rendement_moy,rendement_median,rendement_unite,destination_id,destination,prixreel,prixreel_unite,prixref,prixref_unite,prixref_campagnes,composant_culture_code,action_id,libelle
id,,,,,,,,,,,,,,,,,,,
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_a4e402d1-668e-413f-b0f2-273d8d941c76,"Trèfle blanc,,",100,0,0,NaN,NaN,0.666667,NaN,TONNE_MS_HA,fr.inra.agrosyst.api.entities.referential.RefD...,Pâturage,NaN,EURO_T,NaN,NaN,NaN,d45e4bc7-7554-434e-8c94-368e15d80dc2,fr.inra.agrosyst.api.entities.action.AbstractA...,Pâturage
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_31e3ca0a-328b-4c20-86e8-7cb6459b6889,"Navet,Potager,",100,0,0,NaN,NaN,0.000000,NaN,UNITE_HA,fr.inra.agrosyst.api.entities.referential.RefD...,Toutes catégories - tous conditionnements,NaN,EURO_HA,NaN,NaN,NaN,bb194d7d-e280-4849-84dc-dfefb7e47e10,fr.inra.agrosyst.api.entities.action.AbstractA...,Toutes catégories - tous conditionnements
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_11ecdd07-d103-4df5-998b-96092c994366,"Oignon,Sec,",100,0,0,NaN,NaN,32.000000,NaN,TONNE_HA,fr.inra.agrosyst.api.entities.referential.RefD...,Toutes catégories - tous calibres,1.64,EURO_KG,NaN,NaN,NaN,ad0f5973-a9db-4579-be06-63f54ac3e56f,fr.inra.agrosyst.api.entities.action.AbstractA...,Toutes catégories - tous calibres
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_e9fb93fa-8ffc-4530-b9f6-85b3672cbae4,"Vigne,,",100,0,0,NaN,NaN,11000.000000,NaN,KG_RAISIN_HA,fr.inra.agrosyst.api.entities.referential.RefD...,Toutes appellations / Tous modes de commercial...,NaN,EURO_HA,NaN,NaN,NaN,9521a664-6c41-4a6d-883c-43691c2eb6fe,fr.inra.agrosyst.api.entities.action.AbstractA...,Toutes appellations / Tous modes de commercial...
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_54889d63-6928-4317-93c0-e1828680582b,"Oignon,Sec,",100,0,0,NaN,NaN,10.000000,NaN,TONNE_HA,fr.inra.agrosyst.api.entities.referential.RefD...,Toutes catégories - tous calibres,NaN,EURO_HA,NaN,NaN,NaN,ca4d62b3-2c0f-42b4-bafd-6f3321847794,fr.inra.agrosyst.api.entities.action.AbstractA...,Toutes catégories - tous calibres
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_ffff30ba-1b11-4ed7-957e-c01d5382edd6,"Maïs,,",100,0,0,NaN,NaN,20.833333,NaN,Q_HA_TO_STANDARD_HUMIDITY,fr.inra.agrosyst.api.entities.referential.RefD...,Grain,NaN,EURO_T,22.99990,EURO_Q,2022.0,e7486084-589c-4990-94c0-65cb89465c21,fr.inra.agrosyst.api.entities.action.Harvestin...,Grain
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_ffff509d-6d86-47d9-9a8e-0b64b15907e1,"Ray-grass anglais,,",0,100,0,NaN,NaN,1.000000,NaN,TONNE_MS_HA,fr.inra.agrosyst.api.entities.referential.RefD...,Pâturage,NaN,EURO_T,177.55296,EURO_T,2017.0,d9289427-d373-48a1-bd34-e9882227d9b3,fr.inra.agrosyst.api.entities.action.AbstractA...,Pâturage
fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_ffffc1ab-db1c-4965-a1be-68d0143c7129,"Vigne,,",100,0,0,NaN,NaN,25.000000,NaN,HL_VIN_HA,fr.inra.agrosyst.api.entities.referential.RefD...,Toutes appellations / Tous modes de commercial...,NaN,EURO_HL_WINE,200.00000,EURO_HL_WINE,2023.0,3bda87bf-7aee-40d8-b844-f9592c4525b3,fr.inra.agrosyst.api.entities.action.Harvestin...,Toutes appellations / Tous modes de commercial...
